# Mini-VLA Maze: inspect, run, train, evaluate

The default workflow uses the included checkpoint. Change the experiment settings below to retrain. The source modules in `src/mini_vla_maze` are the single implementation used by the CLI and this notebook.

**Local:** open this notebook inside the extracted project. **Colab:** upload this project's ZIP when the setup cell asks for it. No model-hub login is required.

In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys, zipfile
candidates = [Path.cwd(), *Path.cwd().parents, *Path.cwd().glob('mini-vla-maze*')]
ROOT = next((p for p in candidates if (p/'pyproject.toml').exists() and (p/'src/mini_vla_maze').exists()), None)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Extract the project ZIP, then open this notebook from its notebooks directory.')
    print('Upload mini-vla-maze.zip')
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.zip'):
            destination=Path.cwd().resolve()
            with zipfile.ZipFile(name) as archive:
                for entry in archive.infolist():
                    resolved=(destination/entry.filename).resolve()
                    if not resolved.is_relative_to(destination): raise ValueError('Unsafe archive path')
                archive.extractall(destination)
    ROOT=next((p.parent for p in Path.cwd().glob('*/pyproject.toml') if (p.parent/'src/mini_vla_maze').exists()), None)
    if ROOT is None: raise RuntimeError('The uploaded ZIP did not contain this project.')
ROOT=ROOT.resolve()
packages={'torch':'torch','numpy':'numpy','PIL':'Pillow','matplotlib':'matplotlib','IPython':'ipython'}
missing=[p for mod,p in packages.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
sys.path.insert(0,str(ROOT/'src'))
import json, torch
from IPython.display import display, Image, Markdown
torch.set_num_threads(min(4,torch.get_num_threads()))
print('Project:',ROOT,'| CUDA:',torch.cuda.is_available())

## 1. Experiment settings

`TRAIN=False` and `EVALUATE=False` inspect the supplied reference experiment. Set either flag to run the relevant stage. A fresh output directory avoids reusing cached demonstrations from another configuration.

In [ ]:
TRAIN = False
EVALUATE = False
RUN = ROOT/'runs/notebook'
CHECKPOINT = ROOT/'results/maze/weights.pt'
EPISODES, UPDATES = 320, 1800

## 2. The worlds

White is the pusher; purple is the exit. Orange cells are moving obstacles. Coloured objects are selected through the instruction. Other objects must not be contacted. The rotor alternates between two grid poses, and all mechanisms have a collision interlock.

In [ ]:
import matplotlib.pyplot as plt
from mini_vla_maze.env import make_world, KINDS
from mini_vla_maze.planner import expert
fig,axes=plt.subplots(2,3,figsize=(10,7))
for ax,kind in zip(axes.flat,KINDS):
    world=next(w for seed in range(900,1900) if (w:=make_world(seed,kind)) and expert(w) is not None)
    ax.imshow(world.render()); ax.set_title(kind); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Architecture and reference training

Direct control uses the learned action logits. Hybrid control uses learned attention and semantic labels with A*. The hybrid's route is not produced by the neural action head.

In [ ]:
display(Image(filename=str(ROOT/'media/paper_architecture.png')))
reference=json.loads((ROOT/'results/maze/training.json').read_text())
print({k:reference[k] for k in ['parameters','samples','best_step','train_seconds','validation_after']})
display(Image(filename=str(ROOT/'media/learning.png')))

## 4. Train from demonstrations (optional)

Data is generated by the privileged A* teacher. Checkpoint selection uses validation loss only. The held-out test scenes are not used for optimisation.

In [ ]:
if TRAIN:
    from mini_vla_maze.train import run as train
    report=train(RUN,episodes=EPISODES,steps=UPDATES,seed=17)
    CHECKPOINT=RUN/'weights.pt'
else:
    print('Using the included reference checkpoint:',CHECKPOINT)

## 5. Run a policy

Change `KIND` and `CONTROLLER` to compare conditions. The first solvable scene at or after `SEED` is used and its actual seed is printed. Set `EXECUTE=4` for open-loop action chunks in the direct policy. Hybrid planning always observes each step.

In [ ]:
from mini_vla_maze.evaluate import load_model, rollout
from mini_vla_maze.presentation import animation
KIND='tower'                 # open, baffles, gate, sweeper, rotor, tower
CONTROLLER='hybrid'           # direct, hybrid, oracle
EXECUTE=1
SEED=250000
model=load_model(CHECKPOINT)
world=next(w for seed in range(SEED,SEED+1000) if (w:=make_world(seed,KIND)) and expert(w) is not None)
row,frames=rollout(model,world,CONTROLLER,EXECUTE,record=True)
RUN.mkdir(parents=True,exist_ok=True)
animation(frames,row,RUN/'demo.gif')
print(row)
display(Image(filename=str(RUN/'demo.gif')))

## 6. Same pixels, another instruction

Change the commanded object while keeping the initial image fixed. This is a positive counterfactual demonstration, unlike scoring a wrong instruction against the original target.

In [ ]:
import numpy as np
candidates=[]
for target in range(len(world.objects)):
    alternative=world.clone(); alternative.target=target
    if target!=world.target and expert(alternative) is not None: candidates.append(alternative)
if candidates:
    alternative=candidates[0]
    assert np.array_equal(world.render(),alternative.render())
    row2,frames2=rollout(model,alternative,'hybrid',record=True)
    animation(frames2,row2,RUN/'alternative.gif')
    print(row2); display(Image(filename=str(RUN/'alternative.gif')))
else:
    print('No second solvable command in this scene. See the paired examples in GALLERY.md.')

## 7. Evaluate and inspect failures

Sixteen solvable scenes per primary suite, shared across controllers. The returned intervals are conditional on one trained model. They do not include training-seed variability.

In [ ]:
if EVALUATE:
    from mini_vla_maze.evaluate import run as evaluate
    metrics=evaluate(CHECKPOINT,RUN/'eval',n=16)
else:
    metrics=json.loads((ROOT/'results/maze/metrics.json').read_text())
for key,value in metrics.items():
    print(f"{key:35} {value['successes']:2}/{value['episodes']}  {value['rate']:6.1%}")
display(Image(filename=str(ROOT/'media/benchmark_illustrated.png')))
display(Image(filename=str(ROOT/'media/stress.png')))
print('The plots above are the packaged reference results; new evaluation scores are printed separately.')

## 8. Export your run

Keep `training.json`, `weights.pt`, the data manifests and the episode-level CSV with each result. The generated datasets can be regenerated and need not be committed. The repository report explains the information boundaries, limitations and relation to the continuous baseline.

In [ ]:
import shutil
if RUN.exists():
    archive=shutil.make_archive(str(RUN.parent/'notebook_results'),'zip',RUN)
    print('Saved:',archive)
display(Markdown('Read [methods](../docs/METHODS.md), [results](../RESULTS.md) and [gallery](../GALLERY.md).'))